# Load Inference Results JSON → Silver Table

Reads `inference_results_silver.json` (output from `Inference-GLCReconV9` cell 20)
and writes it to `demo.silver_layer.perception_results`.

**Run this on Databricks** where `spark` is available.

In [ ]:
# %% Config
from pyspark.sql import functions as F

CATALOG     = "dbw_ted_hackathon_dev_001"
GOLD_TABLE = f"{CATALOG}.gold_layer.perception_results"

# Adjust path to where cells 1-18 of Inference-GLCReconV9 generated it
JSON_PATH = "dbfs:/FileStore/inference_results_silver.json"

print(f"Gold table: {GOLD_TABLE}")
print(f"JSON source : {JSON_PATH}")

In [ ]:
# %% Read JSON into Spark DataFrame
df = spark.read.json(JSON_PATH, multiLine=True)

print(f"Rows loaded: {df.count()}")
print(f"Columns: {df.columns}")
df.printSchema()
df.show(5, truncate=40)

In [ ]:
# %% HOTFIX: Compute incremental observed_at from row ordering
# The inference JSON lacks observed_at. We derive it from robot_code grouping
# with deterministic 70-second spacing (64s cycle + 6s sleep) per robot.
#
# Timeline logic:
#   - Each robot's cycles are 70 seconds apart within a day
#   - Robots are staggered by days: R-00 starts day 0, R-01 day 7, etc.
#   - This ensures the simulation timeline spans multiple distinct hours.

from datetime import datetime, timedelta
from pyspark.sql import Row as SparkRow
from pyspark.sql.types import TimestampType

# ── Collect all rows and compute observed_at ────────────────────────────────
all_rows = df.collect()
print(f"Collected {len(all_rows)} rows for observed_at assignment")

# Base time: 30 days ago from now, midnight UTC
base_time = datetime(2025, 7, 1, 6, 0, 0)  # 2025-07-01 06:00:00

# Group rows by robot_code preserving original order within each group
robot_rows: dict[str, list] = {}
for row in all_rows:
    rc = row.robot_code
    if rc not in robot_rows:
        robot_rows[rc] = []
    robot_rows[rc].append(row)

# Stagger robots by days (each robot gets a different start day offset)
robot_list = sorted(robot_rows.keys())
STAGGER_DAYS = 7  # days between robot start times
CYCLE_SPACING_SEC = 70  # 64s cycle + 6s sleep

rows_with_time = []
for robot_idx, robot_code in enumerate(robot_list):
    rows_for_robot = robot_rows[robot_code]
    # Each robot starts STAGGER_DAYS after the previous one
    robot_base = base_time + timedelta(days=robot_idx * STAGGER_DAYS)

    for cycle_idx, row in enumerate(rows_for_robot):
        # Advance 70 seconds per cycle within the robot's operating day
        observed_at = robot_base + timedelta(seconds=cycle_idx * CYCLE_SPACING_SEC)
        row_dict = row.asDict()
        row_dict["observed_at"] = observed_at
        rows_with_time.append(row_dict)

print(f"Assigned incremental observed_at to {len(rows_with_time)} rows")
print(f"  Robots: {robot_list}")
print(f"  First observed_at: {rows_with_time[0]['observed_at']}")
print(f"  Last observed_at:  {rows_with_time[-1]['observed_at']}")

# ── Reconstruct DataFrame with observed_at ──────────────────────────────────
spark_rows = [SparkRow(**r) for r in rows_with_time]
df = spark.createDataFrame(spark_rows)
print(f"  Schema now includes observed_at: {'observed_at' in df.columns}")
df.select("robot_code", "observed_at").show(10, truncate=False)

In [ ]:
# %% Create silver table and write
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver_layer")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {GOLD_TABLE} (
    event_id          STRING,
    robot_code        STRING,
    program_number    BIGINT,
    observed_at       TIMESTAMP,
    abnormal_type     STRING,
    abnormal_score    DOUBLE,
    severity_hint     STRING,
    is_file_anomaly   BOOLEAN,
    per_sensor_scores STRING
)
USING DELTA
""")

# Drop and re-populate
spark.sql(f"DELETE FROM {GOLD_TABLE}")

# HOTFIX: Generate unique event_id from robot_code + observed_at
# (the original JSON had event_id = start_time, which was the same for all rows)
df_clean = df.select(
    F.concat(F.col("robot_code"), F.lit("-"), F.col("observed_at").cast("string")).alias("event_id"),
    "robot_code",
    "program_number",
    "observed_at",
    "abnormal_type",
    "abnormal_score",
    "severity_hint",
    "is_file_anomaly",
    F.to_json("per_sensor_scores").alias("per_sensor_scores"),
)

df_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(GOLD_TABLE)

cnt = spark.table(GOLD_TABLE).count()
print(f"  \u2713 {cnt} rows written to {GOLD_TABLE}")

In [ ]:
# %% Verify
print("=== Robots summary ===")
spark.sql(f"""
SELECT robot_code,
       COUNT(*) AS total,
       SUM(CASE WHEN is_file_anomaly THEN 1 ELSE 0 END) AS anomalies,
       MIN(observed_at) AS first_event,
       MAX(observed_at) AS last_event
FROM {GOLD_TABLE}
GROUP BY robot_code
ORDER BY robot_code
""").show(truncate=False)

# Confirm timeline spans multiple distinct hours (critical for simulation)
print("=== Timeline span ===")
spark.sql(f"""
SELECT
    MIN(observed_at) AS timeline_start,
    MAX(observed_at) AS timeline_end,
    COUNT(DISTINCT date_trunc('HOUR', observed_at)) AS distinct_hours
FROM {GOLD_TABLE}
""").show(truncate=False)

print("=== Sample rows with observed_at ===")
spark.sql(f"""
SELECT robot_code, observed_at, abnormal_type, abnormal_score
FROM {GOLD_TABLE}
ORDER BY observed_at
LIMIT 10
""").show(truncate=False)

In [ ]:
# %% [Analysis A1] Load silver table for threshold tuning
# ─────────────────────────────────────────────────────────────
# L1 = burst detection  : density of is_file_anomaly==True events per hour
#                         fires when  event_count / window_hours >= density_threshold
# L2 = slope detection  : linear regression on daily-avg abnormal_score over days
#                         fires when  slope >= min_slope  (trend = "escalating")
#
# Goal: ≤ 10 triggers per robot over the 8 000-cycle test timeline.
# ─────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np

df = spark.table(GOLD_TABLE).toPandas()
df["observed_at"] = pd.to_datetime(df["observed_at"])
df = df.sort_values(["robot_code", "observed_at"]).reset_index(drop=True)

robots = sorted(df["robot_code"].unique())

print(f"Total rows  : {len(df):,}")
print(f"Robots      : {robots}")
print()

# Per-robot summary
rows = []
for r in robots:
    sub = df[df["robot_code"] == r]
    span_h = (sub["observed_at"].max() - sub["observed_at"].min()).total_seconds() / 3600
    n_anom = sub["is_file_anomaly"].sum()
    rows.append({
        "robot": r,
        "cycles": len(sub),
        "span_hours": round(span_h, 1),
        "anomaly_events": int(n_anom),
        "anomaly_rate_%": round(100 * n_anom / len(sub), 2),
        "score_mean": round(sub["abnormal_score"].mean(), 4),
        "score_max": round(sub["abnormal_score"].max(), 4),
    })

overview = pd.DataFrame(rows).set_index("robot")
print("=== Per-robot overview ===")
print(overview.to_string())
print()
print("NOTE: L1 looks at is_file_anomaly=True density; L2 looks at daily avg abnormal_score slope.")

In [ ]:
# %% [Analysis A2] L1 burst detection simulation
# Mirrors detect_l1_burst: at each observed_at timestamp t for robot r,
#   count is_file_anomaly==True events in (t - window_hours, t]
#   density_per_hour = count / window_hours
#   trigger = density_per_hour >= density_threshold
#
# We sweep density_threshold and count edge-triggers (False→True transitions)
# to see how many distinct L1 burst incidents arise per robot.
# ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

WINDOW_HOURS = 1  # same as get_l1_summary default single-window

def simulate_l1(sub: pd.DataFrame, window_hours: float, density_threshold: float) -> int:
    """Return number of edge-triggered L1 bursts for one robot."""
    times = sub["observed_at"].values
    is_anom = sub["is_file_anomaly"].values.astype(bool)
    window_ns = np.timedelta64(int(window_hours * 3600e9), "ns")

    prev_state = False
    edge_count = 0
    for i, t in enumerate(times):
        # count anomalies in (t - window, t]
        mask = (times <= t) & (times > t - window_ns)
        cnt = is_anom[mask].sum()
        density = cnt / window_hours
        state = density >= density_threshold
        if state and not prev_state:
            edge_count += 1
        prev_state = state
    return edge_count

# --- sweep density_threshold ---
d_thresholds = np.round(np.arange(0.5, 20.1, 0.5), 1)
l1_results = {}  # {robot: [trigger_count per threshold]}

for r in robots:
    sub = df[df["robot_code"] == r].reset_index(drop=True)
    counts = [simulate_l1(sub, WINDOW_HOURS, t) for t in d_thresholds]
    l1_results[r] = counts

TARGET = 10
# max triggers across robots for each threshold
max_per_threshold = [max(l1_results[r][i] for r in robots) for i in range(len(d_thresholds))]

# suggested L1: lowest threshold where worst robot still ≤ TARGET
valid = [(t, mx) for t, mx in zip(d_thresholds, max_per_threshold) if mx <= TARGET]
suggested_L1_density = valid[0][0] if valid else None

# --- plot 1: max triggers vs threshold ---
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
ax.plot(d_thresholds, max_per_threshold, color="steelblue", lw=2, marker="o", ms=4)
ax.axhline(TARGET, color="red", linestyle="--", lw=1.5, label=f"Target ≤ {TARGET}")
if suggested_L1_density:
    ax.axvline(suggested_L1_density, color="green", linestyle="--", lw=1.5,
               label=f"Suggested density_threshold = {suggested_L1_density}")
ax.set_xlabel("density_threshold (events / hour)")
ax.set_ylabel("Max L1 edge-triggers (worst robot)")
ax.set_title(f"L1 Burst — {WINDOW_HOURS}h window sweep")
ax.legend(fontsize=9)
ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

# --- plot 2: per-robot trigger count at suggested threshold ---
ax2 = axes[1]
if suggested_L1_density:
    idx = list(d_thresholds).index(suggested_L1_density)
    per_robot_at_suggested = [l1_results[r][idx] for r in robots]
    ax2.barh(robots, per_robot_at_suggested, color="steelblue")
    ax2.axvline(TARGET, color="red", linestyle="--", lw=1.5, label=f"Target ≤ {TARGET}")
    ax2.set_xlabel("L1 edge-triggers")
    ax2.set_title(f"L1 triggers per robot @ density_threshold={suggested_L1_density}")
    ax2.legend(fontsize=9)

plt.tight_layout()
display(fig)
plt.close()

# --- detailed table at suggested threshold ---
print(f"\nSuggested L1 density_threshold : {suggested_L1_density} events/hour  ({WINDOW_HOURS}h window)")
print()
if suggested_L1_density:
    idx = list(d_thresholds).index(suggested_L1_density)
    print(f"{'robot':<12} {'L1 triggers':>12}")
    print("-" * 26)
    for r in robots:
        print(f"{r:<12} {l1_results[r][idx]:>12}")
    print(f"{'MAX':>12} {max(l1_results[r][idx] for r in robots):>12}")

In [ ]:
# %% [Analysis A3] L2 slope detection — anomaly RATE trend per 6h bucket
# Logic: group into 6h buckets → anomaly_rate = count(is_file_anomaly=True) / total_cycles
# Linear regression on rate values → slope = change in anomaly_rate per 6h bucket
# trend = "escalating" if slope >= min_slope
# Count edge-triggers (non-escalating → escalating transitions)
# ─────────────────────────────────────────────────────────────

BUCKET_HOURS = 6
BUCKETS_WINDOW = 8      # 8 × 6h = 48h rolling window
MIN_BUCKETS_FOR_REG = 3


def _linreg_slope(y_vals):
    """OLS slope + R² on 0-indexed x."""
    n = len(y_vals)
    x = list(range(n))
    x_m = sum(x) / n
    y_m = sum(y_vals) / n
    num = sum((xi - x_m) * (yi - y_m) for xi, yi in zip(x, y_vals))
    den = sum((xi - x_m) ** 2 for xi in x)
    slope = num / den if den else 0.0
    ss_tot = sum((y - y_m) ** 2 for y in y_vals)
    ss_res = sum((y - (y_m + slope * (xi - x_m))) ** 2 for xi, y in zip(x, y_vals))
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
    return slope, r2


def get_rate_buckets(sub: pd.DataFrame) -> pd.Series:
    """Return anomaly_rate indexed by 6h bucket timestamp."""
    sub = sub.copy()
    sub["bucket"] = sub["observed_at"].dt.floor(f"{BUCKET_HOURS}h")
    grp = sub.groupby("bucket")
    rate = grp["is_file_anomaly"].sum() / grp["is_file_anomaly"].count()
    return rate.sort_index()


def simulate_l2_rate(sub: pd.DataFrame, buckets_window: int, min_slope: float) -> int:
    """Return edge-trigger count using anomaly_rate slope."""
    rate = get_rate_buckets(sub)
    vals = rate.tolist()

    prev_escalating = False
    edge_count = 0
    for i in range(len(vals)):
        start = max(0, i + 1 - buckets_window)
        window = vals[start: i + 1]
        if len(window) < MIN_BUCKETS_FOR_REG:
            prev_escalating = False
            continue
        slope, _ = _linreg_slope(window)
        escalating = slope >= min_slope
        if escalating and not prev_escalating:
            edge_count += 1
        prev_escalating = escalating
    return edge_count


# --- visualise anomaly rate per 6h bucket ---
n_robots = len(robots)
fig, axes = plt.subplots(1, n_robots, figsize=(5 * n_robots, 4), squeeze=False)
for i, r in enumerate(robots):
    sub = df[df["robot_code"] == r]
    rate = get_rate_buckets(sub)
    ax = axes[0][i]
    ax.plot(range(len(rate)), rate.values * 100, marker="o", ms=5, color="darkorange")
    ax.set_title(f"{r}  ({len(rate)} buckets × {BUCKET_HOURS}h)", fontsize=9)
    ax.set_xlabel(f"{BUCKET_HOURS}h bucket index")
    ax.set_ylabel("anomaly rate (%)")
    ax.set_ylim(0, 100)
    ax.axhline(rate.mean() * 100, color="gray", linestyle="--", lw=1,
               label=f"mean={rate.mean()*100:.1f}%")
    ax.legend(fontsize=7)

plt.suptitle(f"Anomaly rate per {BUCKET_HOURS}h bucket — L2 slope input", fontsize=11)
plt.tight_layout()
display(fig)
plt.close()

# --- sweep min_slope ---
# slope units: change in anomaly_rate (0–1 scale) per 6h bucket
slope_thresholds = np.round(np.arange(0.001, 0.201, 0.002), 4)
l2_results = {}
for r in robots:
    sub = df[df["robot_code"] == r].reset_index(drop=True)
    l2_results[r] = [simulate_l2_rate(sub, BUCKETS_WINDOW, t) for t in slope_thresholds]

max_per_slope = [max(l2_results[r][i] for r in robots) for i in range(len(slope_thresholds))]
valid_slope = [(t, mx) for t, mx in zip(slope_thresholds, max_per_slope) if mx <= TARGET]
suggested_L2_slope = valid_slope[0][0] if valid_slope else None

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(slope_thresholds, max_per_slope, color="darkorange", lw=2, marker="o", ms=3)
ax.axhline(TARGET, color="red", linestyle="--", lw=1.5, label=f"Target ≤ {TARGET}")
if suggested_L2_slope:
    ax.axvline(suggested_L2_slope, color="green", linestyle="--", lw=1.5,
               label=f"Suggested min_slope = {suggested_L2_slope}")
ax.set_xlabel(f"min_slope  (anomaly_rate change per {BUCKET_HOURS}h bucket,  scale 0–1)")
ax.set_ylabel("Max L2 edge-triggers (worst robot)")
ax.set_title(f"L2 Rate-Slope — rolling {BUCKETS_WINDOW}×{BUCKET_HOURS}h window sweep")
ax.legend(fontsize=9)
ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))
plt.tight_layout()
display(fig)
plt.close()

print(f"\nSuggested L2 min_slope : {suggested_L2_slope}")
print(f"  (anomaly_rate change per {BUCKET_HOURS}h bucket, rolling {BUCKETS_WINDOW} buckets)")
if suggested_L2_slope:
    idx = list(slope_thresholds).index(suggested_L2_slope)
    print(f"\n{'robot':<12} {'L2 triggers':>12}  {'rate_mean':>10}  {'rate_std':>10}")
    print("-" * 48)
    for r in robots:
        rate = get_rate_buckets(df[df["robot_code"] == r])
        print(f"{r:<12} {l2_results[r][idx]:>12}  {rate.mean()*100:>9.1f}%  {rate.std()*100:>9.1f}%")
    print(f"{'MAX':>12} {max(l2_results[r][idx] for r in robots):>12}")
else:
    print(f"  All robots have < {MIN_BUCKETS_FOR_REG} buckets — L2 N/A.")

In [ ]:
# %% [Analysis A4] Final recommendation — config block to copy into poller / engine
print("=" * 62)
print("THRESHOLD RECOMMENDATION  (goal: ≤ 10 triggers / robot)")
print("=" * 62)
print()
print("L1 — burst detection (detect_l1_burst / CognitivePoller)")
print(f"  density_threshold  = {suggested_L1_density}  events / hour")
print(f"  window_hours       = {WINDOW_HOURS}")
print()
print("L2 — slope detection (detect_l2_slope, 6h-bucket variant)")
if suggested_L2_slope:
    print(f"  min_slope          = {suggested_L2_slope}  avg_score / {BUCKET_HOURS}h bucket")
else:
    print(f"  min_slope          = 0.015  (default — L2 had insufficient buckets on this dataset)")
print(f"  buckets_window     = {BUCKETS_WINDOW} × {BUCKET_HOURS}h")
print()
print("─" * 62)
print("Copy-paste config:")
print(f"""
# CognitivePoller / engine — tuned on 8 000-cycle test set
L1_DENSITY_THRESHOLD = {suggested_L1_density}    # events / hour, {WINDOW_HOURS}h window
L2_MIN_SLOPE         = {suggested_L2_slope if suggested_L2_slope else 0.015}   # avg_score change / {BUCKET_HOURS}h bucket
""")

# Sanity check: top-10 densest 1h windows
print("=== Top 10 densest 1-h windows (would fire at L1) ===")
records = []
for r in robots:
    sub = df[df["robot_code"] == r]
    times = sub["observed_at"].values
    is_anom = sub["is_file_anomaly"].values.astype(bool)
    window_ns = np.timedelta64(int(WINDOW_HOURS * 3600e9), "ns")
    for t, anomaly in zip(times, is_anom):
        mask = (times <= t) & (times > t - window_ns)
        density = float(is_anom[mask].sum()) / WINDOW_HOURS
        records.append({"robot": r, "time": str(t)[:19], "density_per_hour": density,
                        "is_file_anomaly_at_t": bool(anomaly)})

top10 = pd.DataFrame(records).sort_values("density_per_hour", ascending=False).head(10)
print(top10.to_string(index=False))